In [26]:
# EX1
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round as spark_round

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

spark = SparkSession.builder.appName('Ex1').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Extract
raw_df = spark.read.csv('/content/drive/MyDrive/ETL_Lab/sales_data.csv', header=True, inferSchema=True)

print('=== Raw Data (first 10 rows) ===')
raw_df.show(10)
print('Total records:', raw_df.count())
raw_df.printSchema()

# Transform
df_no_nulls = raw_df.fillna({'price': 0.0, 'quantity': 0})

null_count = raw_df.filter(col('price').isNull() | col('quantity').isNull()).count()
print(f'Null rows fixed: {null_count}')

df_deduped = df_no_nulls.dropDuplicates()

removed = df_no_nulls.count() - df_deduped.count()
print(f'Duplicate rows removed: {removed}')
print(f'Records after dedup:    {df_deduped.count()}')

df_final = df_deduped.withColumn(
    'total_amount',
    spark_round(col('price') * col('quantity'), 2)
)

print('=== Final DataFrame with total_amount ===')
df_final.show(10)

# Load
df_final.coalesce(1).write.csv('output/ex1_cleaned', header=True, mode='overwrite')

import os
files = os.listdir('output/ex1_cleaned')
print('Output files:', files)
print('Exercise 1 Complete ✓')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Raw Data (first 10 rows) ===
+--------+----------+-----------+-----+--------+----------+
|order_id|   product|   category|price|quantity|order_date|
+--------+----------+-----------+-----+--------+----------+
|    1001|    Tablet|Electronics|  802|       2|2025-03-01|
|    1002|Headphones|Accessories|  501|       3|2025-03-02|
|    1003|    Laptop|Electronics|  829|       2|2025-03-03|
|    1004|    Tablet|Accessories|  655|       1|2025-03-04|
|    1005|    Tablet|Electronics|  261|       4|2025-03-05|
|    1006|Headphones|Accessories|  301|       3|2025-03-06|
|    1007|    Laptop|Accessories|  369|       1|2025-03-07|
|    1008|    Laptop|Electronics|  962|       4|2025-03-08|
|    1009|    Tablet|Electronics|  915|       4|2025-03-09|
|    1010|     Phone|Electronics|  370|       2|2025-03-10|
+--------+----------+-----------+-----+--------+----------

In [27]:
# EX 2
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, to_date

from google.colab import drive
drive.mount('/content/drive')

spark = SparkSession.builder.appName('Ex2').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Read the JSON File
raw_df = spark.read.json('/content/drive/MyDrive/ETL_Lab/user_logs.json')

print('=== Raw JSON (Spark auto-creates nested struct column) ===')
raw_df.show(truncate=False)
raw_df.printSchema()

# Flatten Nested Fields
df_flat = raw_df.select(
    col('user_id'),
    col('activity.type').alias('activity_type'),
    col('activity.timestamp').alias('raw_timestamp')
)

print('=== Flattened DataFrame ===')
df_flat.show(truncate=False)

# Convert Timestamp String to Proper Types
df_ts = df_flat \
    .withColumn('event_timestamp',
        to_timestamp(col('raw_timestamp'), "yyyy-MM-dd'T'HH:mm:ss")) \
    .withColumn('event_date',
        to_date(col('raw_timestamp'), "yyyy-MM-dd'T'HH:mm:ss")) \
    .drop('raw_timestamp')   # remove original string column

print('=== After Timestamp Conversion ===')
df_ts.show(truncate=False)
df_ts.printSchema()

# Filter Only Login Events
df_logins = df_ts.filter(col('activity_type') == 'login')

print(f'Total records: {df_ts.count()}')
print(f'Login records only: {df_logins.count()}')
print('=== Login Events Only ===')
df_logins.show(truncate=False)

# Write to Parquet
df_logins.coalesce(1).write.parquet('output/ex2_logins', mode='overwrite')

verify_df = spark.read.parquet('output/ex2_logins')
verify_df.printSchema()
print(f'Records read back from Parquet: {verify_df.count()}')
print('Exercise 2 Complete ✓')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== Raw JSON (Spark auto-creates nested struct column) ===
+---------------+-------------------------------+-------+
|_corrupt_record|activity                       |user_id|
+---------------+-------------------------------+-------+
|{              |NULL                           |NULL   |
|  "logs": [    |NULL                           |NULL   |
|NULL           |{2025-03-10T10:00:00, login}   |101    |
|NULL           |{2025-03-10T10:30:00, logout}  |102    |
|NULL           |{2025-03-10T11:00:00, login}   |103    |
|NULL           |{2025-03-10T12:00:00, purchase}|104    |
|NULL           |{2025-03-11T09:00:00, login}   |105    |
|NULL           |{2025-03-11T10:00:00, login}   |106    |
|NULL           |{2025-03-11T11:00:00, logout}  |107    |
|NULL           |{2025-03-12T08:30:00, login}   |108    |
|  ]            |NULL                           |NULL   |


In [25]:
# EX3
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg as spark_avg
from pyspark.sql.functions import round as spark_round, desc

from google.colab import drive
drive.mount('/content/drive')

spark = SparkSession.builder.appName('Ex3').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

df = spark.read.csv('/content/drive/MyDrive/ETL_Lab/sales_data.csv', header=True, inferSchema=True) \
    .fillna({'price': 0.0, 'quantity': 0}) \
    .dropDuplicates() \
    .withColumn('total_amount', spark_round(col('price') * col('quantity'), 2))

df.createOrReplaceTempView('sales')
print(f'Base data ready: {df.count()} records registered as SQL view [sales]')

revenue = df.groupBy('category') \
    .agg(spark_round(spark_sum('total_amount'), 2).alias('total_revenue')) \
    .orderBy(desc('total_revenue'))

# TASK A
# Identical query using Spark SQL syntax:
revenue_sql = spark.sql('''
    SELECT category,
           ROUND(SUM(price * quantity), 2) AS total_revenue
    FROM   sales
    GROUP BY category
    ORDER BY total_revenue DESC
''')

print('=== Task A: Revenue by Category ===')
revenue.show()

# TASK B
top_products = df.groupBy('product') \
    .agg(spark_sum('quantity').alias('total_qty')) \
    .orderBy(desc('total_qty')) \
    .limit(3)

print('=== Task B: Top 3 Products by Quantity ===')
top_products.show()

# TASK C
avg_daily = df.groupBy('order_date') \
    .agg(spark_round(spark_avg('total_amount'), 2).alias('avg_order_value')) \
    .orderBy('order_date')

print('=== Task C: Average Order Value per Day ===')
avg_daily.show(10)

# Save All Results
revenue.write.mode('overwrite').parquet('output/ex3_revenue_by_category')
top_products.write.mode('overwrite').parquet('output/ex3_top_products')
avg_daily.write.mode('overwrite').parquet('output/ex3_avg_per_day')

print('All 3 aggregation results saved as Parquet ✓')
print('Exercise 3 Complete ✓')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base data ready: 50 records registered as SQL view [sales]
=== Task A: Revenue by Category ===
+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|Accessories|        38234|
|Electronics|        36683|
+-----------+-------------+

=== Task B: Top 3 Products by Quantity ===
+----------+---------+
|   product|total_qty|
+----------+---------+
|Headphones|       41|
|    Tablet|       36|
|     Phone|       27|
+----------+---------+

=== Task C: Average Order Value per Day ===
+----------+---------------+
|order_date|avg_order_value|
+----------+---------------+
|2025-03-01|         1604.0|
|2025-03-02|         1503.0|
|2025-03-03|         1658.0|
|2025-03-04|          655.0|
|2025-03-05|         1044.0|
|2025-03-06|          903.0|
|2025-03-07|          369.0|
|2025-03-08|         3848.0|
|2025-03-09|         3660.0|
|2025-03-1

In [29]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, round as spark_round
import os, glob

from google.colab import drive
drive.mount('/content/drive')

spark = SparkSession.builder.appName('Ex4').master('local[*]').getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

FINAL_OUTPUT = 'output/ex4_partitioned'

# STEP 1: Simulate file tracking
already_processed = {
    '/content/drive/MyDrive/ETL_Lab/sales_2025_03_01.csv',
    '/content/drive/MyDrive/ETL_Lab/sales_2025_03_02.csv',
    '/content/drive/MyDrive/ETL_Lab/sales_2025_03_03.csv',
}

# STEP 2: Find all daily files and filter to NEW ones only
all_files = sorted(glob.glob('/content/drive/MyDrive/ETL_Lab/sales_2025_03_*.csv'))
new_files  = [f for f in all_files if f not in already_processed]

print(f'All files found  : {len(all_files)}')
print(f'Already processed: {len(already_processed)}')
print(f'New files today  : {len(new_files)}')
for f in new_files:
    print(f'  → {f}')

# STEP 3: Read ONLY the new files
if new_files:
    new_df = spark.read.csv(new_files, header=True, inferSchema=True) \
        .withColumn('total_amount', spark_round(col('price') * col('quantity'), 2))

    print(f'\nNew records loaded: {new_df.count()}')
    new_df.show()

    # STEP 4: Append to existing dataset (if it exists)
    if os.path.exists(FINAL_OUTPUT):
        existing_df = spark.read.parquet(FINAL_OUTPUT)
        print(f'Existing records : {existing_df.count()}')
        combined_df = existing_df.union(new_df)   # stack rows vertically
    else:
        print('First run — no existing dataset')
        combined_df = new_df

    # STEP 5: Deduplicate using order_id as unique key
    final_df = combined_df.dropDuplicates(['order_id'])
    print(f'Combined (before dedup): {combined_df.count()}')
    print(f'Final    (after  dedup): {final_df.count()}')

    # STEP 6: Write partitioned Parquet
    final_df.write \
        .partitionBy('order_date') \
        .mode('overwrite') \
        .parquet(FINAL_OUTPUT)

    print('\nPartitions created:')
    for d in sorted(os.listdir(FINAL_OUTPUT)):
        if d.startswith('order_date='):
            print(f'  {d}/')
    print('Exercise 4 Complete ✓')
elif os.path.exists(FINAL_OUTPUT):
    print('No new files to process, checking existing output.')
    existing_df = spark.read.parquet(FINAL_OUTPUT)
    print(f'Existing records : {existing_df.count()}')
    print('Exercise 4 Complete ✓')
else:
    print('No new files to process and no existing dataset. Exercise 4 skipped.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
All files found  : 5
Already processed: 3
New files today  : 2
  → /content/drive/MyDrive/ETL_Lab/sales_2025_03_04.csv
  → /content/drive/MyDrive/ETL_Lab/sales_2025_03_05.csv

New records loaded: 20
+--------+----------+-----------+-----+--------+----------+------------+
|order_id|   product|   category|price|quantity|order_date|total_amount|
+--------+----------+-----------+-----+--------+----------+------------+
|    2040|Headphones|Accessories|  198|       4|2025-03-05|         792|
|    2041|Headphones|Accessories|  758|       4|2025-03-05|        3032|
|    2042|     Phone|Accessories|  915|       3|2025-03-05|        2745|
|    2043|Headphones|Electronics|  307|       1|2025-03-05|         307|
|    2044|    Laptop|Electronics|  230|       4|2025-03-05|         920|
|    2045|Headphones|Accessories|  503|       4|2025-03-05|        2012|
|    2046|Headp